# PPO and RLHF for LLMs

**Module 14: Reinforcement Learning**  
**Objective**: Master advanced RL algorithms and LLM alignment

Advanced RL:
- Proximal Policy Optimization (PPO)
- Generalized Advantage Estimation (GAE)
- RLHF (Reinforcement Learning from Human Feedback)
- Reward Modeling
- InstructGPT Pipeline
- DPO (Direct Preference Optimization)

## What You'll Learn
1. PPO algorithm and clipped objective
2. GAE for variance reduction
3. Reward model training from preferences
4. RLHF for LLM alignment
5. InstructGPT-style pipeline
6. DPO as RLHF alternative

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional, Dict
from collections import deque
from dataclasses import dataclass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. Proximal Policy Optimization (PPO)

**PPO** is the most popular RL algorithm (used in ChatGPT, Claude, Copilot)

**Problem with vanilla Policy Gradient**: Large policy updates can be destructive

**Solution**: Constrain policy update to stay close to old policy

**Clipped Surrogate Objective**:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t[\min(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t)]$$

Where:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ (probability ratio)
- $\hat{A}_t$: Advantage estimate
- $\epsilon$: Clip range (typically 0.1-0.2)

**Intuition**: Clip prevents too-large policy updates

In [ ]:
def visualize_ppo_clipping():
    """Visualize PPO clipped objective."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. Probability Ratio vs Objective
    ax1 = axes[0]
    ax1.set_title('PPO Clipped Objective', fontsize=14, weight='bold')
    
    # Probability ratios
    r = np.linspace(0.5, 2.0, 200)
    epsilon = 0.2
    
    # Positive advantage
    A_pos = 1.0
    objective_pos_unclipped = r * A_pos
    objective_pos_clipped = np.minimum(r * A_pos, (1 + epsilon) * A_pos)
    
    ax1.plot(r, objective_pos_unclipped, 'b--', linewidth=2, label='Unclipped (A>0)', alpha=0.6)
    ax1.plot(r, objective_pos_clipped, 'b-', linewidth=3, label='PPO Clipped (A>0)')
    
    # Negative advantage
    A_neg = -1.0
    objective_neg_unclipped = r * A_neg
    objective_neg_clipped = np.maximum(r * A_neg, (1 - epsilon) * A_neg)
    
    ax1.plot(r, objective_neg_unclipped, 'r--', linewidth=2, label='Unclipped (A<0)', alpha=0.6)
    ax1.plot(r, objective_neg_clipped, 'r-', linewidth=3, label='PPO Clipped (A<0)')
    
    # Clip boundaries
    ax1.axvline(1 - epsilon, color='gray', linestyle=':', linewidth=2, label=f'Clip range [{1-epsilon}, {1+epsilon}]')
    ax1.axvline(1 + epsilon, color='gray', linestyle=':', linewidth=2)
    ax1.axvline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='r=1 (no change)')
    
    ax1.set_xlabel('Probability Ratio r = π_new / π_old', fontsize=12)
    ax1.set_ylabel('Objective Value', fontsize=12)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-1.5, 1.5)
    
    # 2. PPO vs Policy Gradient
    ax2 = axes[1]
    ax2.set_title('PPO vs Vanilla Policy Gradient', fontsize=14, weight='bold')
    ax2.axis('off')
    
    # Create comparison boxes
    box_width = 0.35
    box_height = 0.7
    
    # Vanilla PG box
    rect1 = plt.Rectangle((0.05, 0.15), box_width, box_height, 
                          facecolor='lightcoral', edgecolor='black', linewidth=3)
    ax2.add_patch(rect1)
    ax2.text(0.225, 0.75, 'Vanilla Policy\nGradient', ha='center', va='center',
            fontsize=13, weight='bold')
    
    pg_text = [
        '• High variance',
        '• Unstable training',
        '• Large updates',
        '• Performance collapse',
        '• Sensitive to LR'
    ]
    y_start = 0.62
    for i, text in enumerate(pg_text):
        ax2.text(0.225, y_start - i*0.09, text, ha='center', va='center', fontsize=9)
    
    # PPO box
    rect2 = plt.Rectangle((0.6, 0.15), box_width, box_height, 
                          facecolor='lightgreen', edgecolor='black', linewidth=3)
    ax2.add_patch(rect2)
    ax2.text(0.775, 0.75, 'PPO', ha='center', va='center',
            fontsize=13, weight='bold')
    
    ppo_text = [
        '✓ Lower variance',
        '✓ Stable training',
        '✓ Controlled updates',
        '✓ Robust performance',
        '✓ Few hyperparams'
    ]
    for i, text in enumerate(ppo_text):
        ax2.text(0.775, y_start - i*0.09, text, ha='center', va='center', 
                fontsize=9, color='darkgreen', weight='bold')
    
    # Arrow
    ax2.annotate('', xy=(0.6, 0.5), xytext=(0.42, 0.5),
                arrowprops=dict(arrowstyle='->', lw=4, color='blue'))
    ax2.text(0.51, 0.56, 'Clipping', ha='center', fontsize=11,
            weight='bold', color='blue')
    
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    print("PPO Key Concepts:")
    print("="*70)
    
    print("\nCLIPPING MECHANISM:")
    print("  When advantage A > 0 (good action):")
    print("    • Allow increasing π(a|s), but not beyond (1+ε)·π_old")
    print("    • Prevents over-optimization on single trajectory")
    
    print("\n  When advantage A < 0 (bad action):")
    print("    • Allow decreasing π(a|s), but not below (1-ε)·π_old")
    print("    • Prevents removing action prematurely")
    
    print("\nWHY PPO WORKS:")
    print("  1. Conservative updates: Policy can't change too much")
    print("  2. Multiple epochs: Reuse data with clipping safety")
    print("  3. Simple: No complex trust region constraints (vs TRPO)")
    print("  4. Effective: State-of-the-art on many tasks")
    
    print("\nHYPERPARAMETERS:")
    print(f"  • Clip range ε: 0.1-0.2 (typical: 0.2)")
    print(f"  • Mini-batch size: 64-4096")
    print(f"  • Epochs per update: 3-10")
    print(f"  • GAE λ: 0.95-0.99")

visualize_ppo_clipping()

## 2. Generalized Advantage Estimation (GAE)

**Problem**: Bias-variance tradeoff in advantage estimation

**1-step TD**: Low variance, high bias
$$\hat{A}_t^{(1)} = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**Monte Carlo**: High variance, low bias
$$\hat{A}_t^{(\infty)} = \sum_{k=0}^{\infty} \gamma^k r_{t+k} - V(s_t)$$

**GAE**: Exponentially-weighted average of n-step advantages

$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty}(\gamma\lambda)^l \delta_{t+l}$$

Where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ (TD error)

Parameter $\lambda \in [0,1]$ controls bias-variance:
- $\lambda=0$: GAE = 1-step TD (low variance, high bias)
- $\lambda=1$: GAE = Monte Carlo (high variance, low bias)
- $\lambda=0.95$: Typical choice

In [ ]:
@dataclass
class RolloutBuffer:
    """Buffer for storing rollout data."""
    states: List[np.ndarray]
    actions: List[int]
    rewards: List[float]
    values: List[float]
    log_probs: List[torch.Tensor]
    dones: List[bool]
    
    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.values = []
        self.log_probs = []
        self.dones = []
    
    def add(self, state, action, reward, value, log_prob, done):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.values.append(value)
        self.log_probs.append(log_prob)
        self.dones.append(done)
    
    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.rewards.clear()
        self.values.clear()
        self.log_probs.clear()
        self.dones.clear()
    
    def __len__(self):
        return len(self.states)

def compute_gae(rewards: List[float], values: List[float], dones: List[bool],
                next_value: float, gamma: float = 0.99, lam: float = 0.95) -> Tuple[List[float], List[float]]:
    """
    Compute Generalized Advantage Estimation.
    
    Args:
        rewards: Rewards for each step
        values: Value estimates V(s_t)
        dones: Whether episode ended
        next_value: V(s_{T+1})
        gamma: Discount factor
        lam: GAE lambda
        
    Returns:
        advantages: GAE advantages
        returns: Discounted returns (for value loss)
    """
    advantages = []
    gae = 0
    
    # Backward pass: compute GAE
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value_t = next_value
            next_non_terminal = 1.0 - float(dones[t])
        else:
            next_value_t = values[t + 1]
            next_non_terminal = 1.0 - float(dones[t])
        
        # TD error
        delta = rewards[t] + gamma * next_value_t * next_non_terminal - values[t]
        
        # GAE
        gae = delta + gamma * lam * next_non_terminal * gae
        advantages.insert(0, gae)
    
    # Returns = advantages + values
    returns = [adv + val for adv, val in zip(advantages, values)]
    
    return advantages, returns

# Visualize GAE
print("Generalized Advantage Estimation (GAE):")
print("="*70)

# Example trajectory
rewards_example = [1.0, -0.5, 2.0, 0.5, -1.0]
values_example = [0.5, 0.3, 1.2, 0.8, 0.2]
dones_example = [False, False, False, False, True]
next_value_example = 0.0

# Compute GAE with different lambdas
lambdas = [0.0, 0.5, 0.95, 1.0]

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_title('GAE with Different λ Values', fontsize=14, weight='bold')

for lam in lambdas:
    advantages, _ = compute_gae(rewards_example, values_example, dones_example,
                                next_value_example, gamma=0.99, lam=lam)
    ax.plot(range(len(advantages)), advantages, marker='o', linewidth=2,
           markersize=8, label=f'λ={lam}')

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Advantage Estimate', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

print("\nGAE OBSERVATIONS:")
print("  • λ=0.0: Uses only 1-step TD error (most stable, may miss long-term credit)")
print("  • λ=1.0: Full Monte Carlo (captures all future, highest variance)")
print("  • λ=0.95: Good balance (typical choice in PPO)")
print("  • Higher λ → more variance, less bias")
print("  • Lower λ → less variance, more bias")

print("\nExample trajectory:")
for t in range(len(rewards_example)):
    print(f"  t={t}: reward={rewards_example[t]:+.1f}, value={values_example[t]:.2f}, done={dones_example[t]}")

## 3. PPO Implementation

**Full PPO algorithm**:

1. Collect rollout using policy $\pi_{\theta_{\text{old}}}$
2. Compute advantages using GAE
3. Update policy using clipped objective (multiple epochs)
4. Update value function
5. Repeat

In [ ]:
class PPOActorCritic(nn.Module):
    """
    Actor-Critic for PPO.
    """
    
    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 128):
        super().__init__()
        
        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        
        # Actor (policy)
        self.actor = nn.Linear(hidden_dim, n_actions)
        
        # Critic (value)
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features)
        return logits, value
    
    def get_action_and_value(self, x: torch.Tensor, action: Optional[torch.Tensor] = None) -> Dict:
        """
        Get action, value, log_prob, entropy.
        
        Returns dict with:
            - action: sampled action
            - log_prob: log probability of action
            - entropy: policy entropy
            - value: state value
        """
        logits, value = self(x)
        probs = F.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        
        if action is None:
            action = dist.sample()
        
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        
        return {
            'action': action,
            'log_prob': log_prob,
            'entropy': entropy,
            'value': value.squeeze(-1)
        }

class PPOAgent:
    """
    Proximal Policy Optimization agent.
    """
    
    def __init__(self, state_dim: int, n_actions: int,
                 learning_rate: float = 3e-4,
                 gamma: float = 0.99,
                 gae_lambda: float = 0.95,
                 clip_coef: float = 0.2,
                 value_loss_coef: float = 0.5,
                 entropy_coef: float = 0.01,
                 n_epochs: int = 10,
                 batch_size: int = 64):
        
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_coef = clip_coef
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        
        self.model = PPOActorCritic(state_dim, n_actions).to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        
        self.buffer = RolloutBuffer()
    
    def get_action(self, state: np.ndarray) -> Tuple[int, float, float]:
        """
        Get action from policy.
        
        Returns:
            action: Selected action
            log_prob: Log probability
            value: State value
        """
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = self.model.get_action_and_value(state_t)
        
        return (output['action'].item(),
                output['log_prob'].item(),
                output['value'].item())
    
    def update(self) -> Dict[str, float]:
        """
        PPO update.
        
        Returns:
            Metrics dict
        """
        # Compute advantages
        with torch.no_grad():
            # Next value for last state
            if self.buffer.dones[-1]:
                next_value = 0.0
            else:
                next_state = torch.FloatTensor(self.buffer.states[-1]).unsqueeze(0).to(device)
                _, next_value = self.model(next_state)
                next_value = next_value.item()
        
        advantages, returns = compute_gae(
            self.buffer.rewards,
            self.buffer.values,
            self.buffer.dones,
            next_value,
            self.gamma,
            self.gae_lambda
        )
        
        # Convert to tensors
        states = torch.FloatTensor(np.array(self.buffer.states)).to(device)
        actions = torch.LongTensor(self.buffer.actions).to(device)
        old_log_probs = torch.FloatTensor([lp for lp in self.buffer.log_probs]).to(device)
        advantages_t = torch.FloatTensor(advantages).to(device)
        returns_t = torch.FloatTensor(returns).to(device)
        
        # Normalize advantages
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        
        # Multiple epochs
        metrics = {
            'policy_loss': 0.0,
            'value_loss': 0.0,
            'entropy': 0.0,
            'approx_kl': 0.0,
            'clip_fraction': 0.0
        }
        
        n_updates = 0
        
        for epoch in range(self.n_epochs):
            # Mini-batch updates
            indices = np.arange(len(states))
            np.random.shuffle(indices)
            
            for start in range(0, len(states), self.batch_size):
                end = start + self.batch_size
                batch_indices = indices[start:end]
                
                # Get current policy outputs
                output = self.model.get_action_and_value(
                    states[batch_indices],
                    actions[batch_indices]
                )
                
                # Probability ratio
                log_ratio = output['log_prob'] - old_log_probs[batch_indices]
                ratio = torch.exp(log_ratio)
                
                # Clipped surrogate objective
                adv_batch = advantages_t[batch_indices]
                policy_loss_1 = -adv_batch * ratio
                policy_loss_2 = -adv_batch * torch.clamp(
                    ratio, 1 - self.clip_coef, 1 + self.clip_coef
                )
                policy_loss = torch.max(policy_loss_1, policy_loss_2).mean()
                
                # Value loss
                value_loss = F.mse_loss(output['value'], returns_t[batch_indices])
                
                # Entropy bonus (encourages exploration)
                entropy_loss = output['entropy'].mean()
                
                # Total loss
                loss = policy_loss + self.value_loss_coef * value_loss - self.entropy_coef * entropy_loss
                
                # Optimize
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                self.optimizer.step()
                
                # Metrics
                with torch.no_grad():
                    approx_kl = ((ratio - 1) - log_ratio).mean().item()
                    clip_fraction = ((ratio - 1.0).abs() > self.clip_coef).float().mean().item()
                
                metrics['policy_loss'] += policy_loss.item()
                metrics['value_loss'] += value_loss.item()
                metrics['entropy'] += entropy_loss.item()
                metrics['approx_kl'] += approx_kl
                metrics['clip_fraction'] += clip_fraction
                n_updates += 1
        
        # Average metrics
        for key in metrics:
            metrics[key] /= n_updates
        
        return metrics

print("PPO Implementation:")
print("="*70)

ppo_agent = PPOAgent(state_dim=2, n_actions=4)

print("\nPPO Hyperparameters:")
print(f"  • Learning rate: {3e-4}")
print(f"  • Clip coefficient ε: {ppo_agent.clip_coef}")
print(f"  • GAE λ: {ppo_agent.gae_lambda}")
print(f"  • Value loss coef: {ppo_agent.value_loss_coef}")
print(f"  • Entropy coef: {ppo_agent.entropy_coef}")
print(f"  • Update epochs: {ppo_agent.n_epochs}")
print(f"  • Batch size: {ppo_agent.batch_size}")

print("\nPPO Model:")
print(ppo_agent.model)

## 4. RLHF - Reinforcement Learning from Human Feedback

**RLHF** is the key technique for aligning language models (ChatGPT, Claude, Gemini)

**Problem**: How to make LLMs helpful, harmless, and honest?

**Solution**: Learn from human preferences

**RLHF Pipeline** (InstructGPT-style):

1. **Supervised Fine-tuning (SFT)**: Train on human demonstrations
2. **Reward Modeling (RM)**: Train reward model on human preferences
3. **RL Fine-tuning**: Optimize policy using PPO with learned reward

**Reward Model**:
- Input: (prompt, response)
- Output: Scalar reward (how good is response)
- Trained on pairwise preferences: "Response A is better than B"

In [ ]:
def visualize_rlhf_pipeline():
    """Visualize RLHF pipeline."""
    
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('RLHF Pipeline for LLM Alignment', fontsize=16, weight='bold', pad=20)
    
    # Stage 1: Supervised Fine-tuning
    box1_y = 0.75
    rect1 = plt.Rectangle((0.05, box1_y), 0.9, 0.15, 
                          facecolor='lightblue', edgecolor='black', linewidth=3)
    ax.add_patch(rect1)
    ax.text(0.5, box1_y + 0.12, 'Stage 1: Supervised Fine-Tuning (SFT)',
           ha='center', va='center', fontsize=13, weight='bold')
    
    sft_text = [
        'Input: Pretrained LLM (e.g., GPT-3)',
        'Data: High-quality human demonstrations (prompt → response)',
        'Training: Standard supervised learning (cross-entropy loss)',
        'Output: SFT model with better instruction following'
    ]
    y_pos = box1_y + 0.08
    for text in sft_text:
        ax.text(0.08, y_pos, text, fontsize=9, va='center')
        y_pos -= 0.025
    
    # Arrow to Stage 2
    ax.annotate('', xy=(0.5, box1_y - 0.02), xytext=(0.5, box1_y),
               arrowprops=dict(arrowstyle='->', lw=3, color='black'))
    
    # Stage 2: Reward Modeling
    box2_y = 0.48
    rect2 = plt.Rectangle((0.05, box2_y), 0.9, 0.20,
                          facecolor='lightgreen', edgecolor='black', linewidth=3)
    ax.add_patch(rect2)
    ax.text(0.5, box2_y + 0.17, 'Stage 2: Reward Model Training',
           ha='center', va='center', fontsize=13, weight='bold')
    
    rm_text = [
        'Input: SFT model',
        'Data: Human preferences (prompt, responseA, responseB, preference)',
        '  Example: "Response A is more helpful than Response B"',
        'Model: Reward function r(prompt, response) → scalar',
        'Loss: Bradley-Terry model L = -log(σ(r(x,yₐ) - r(x,yᵦ)))',
        'Output: Reward model predicts human preferences'
    ]
    y_pos = box2_y + 0.13
    for text in rm_text:
        ax.text(0.08, y_pos, text, fontsize=9, va='center')
        y_pos -= 0.025
    
    # Arrow to Stage 3
    ax.annotate('', xy=(0.5, box2_y - 0.02), xytext=(0.5, box2_y),
               arrowprops=dict(arrowstyle='->', lw=3, color='black'))
    
    # Stage 3: RL Fine-tuning
    box3_y = 0.15
    rect3 = plt.Rectangle((0.05, box3_y), 0.9, 0.25,
                          facecolor='lightyellow', edgecolor='black', linewidth=3)
    ax.add_patch(rect3)
    ax.text(0.5, box3_y + 0.22, 'Stage 3: RL Fine-Tuning (PPO)',
           ha='center', va='center', fontsize=13, weight='bold')
    
    rl_text = [
        'Policy: SFT model (to be optimized)',
        'Reward: r(prompt, response) from reward model',
        'KL penalty: Keep policy close to SFT model',
        '  Total reward: r_total = r_RM - β·KL(π_θ || π_SFT)',
        'Algorithm: PPO (Proximal Policy Optimization)',
        'Training: Generate responses, get rewards, update policy',
        'Output: Aligned model (helpful, harmless, honest)'
    ]
    y_pos = box3_y + 0.18
    for text in rl_text:
        ax.text(0.08, y_pos, text, fontsize=9, va='center')
        y_pos -= 0.025
    
    # Final result
    ax.text(0.5, 0.05, '🎉 Result: ChatGPT-style aligned model 🎉',
           ha='center', va='center', fontsize=12, weight='bold',
           bbox=dict(boxstyle='round', facecolor='gold', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    print("\nRLHF Key Insights:")
    print("="*70)
    
    print("\nWHY RLHF WORKS:")
    print("  1. SFT provides good initialization")
    print("  2. Reward model captures human preferences at scale")
    print("  3. PPO optimizes for human preferences while staying safe (KL penalty)")
    
    print("\nKL PENALTY:")
    print("  • Prevents model from exploiting reward model")
    print("  • Keeps output distribution similar to SFT")
    print("  • β typically 0.01-0.1")
    print("  • Without KL: model might output gibberish with high predicted reward")
    
    print("\nCHALLENGES:")
    print("  • Expensive: Requires large-scale human feedback")
    print("  • Reward hacking: Model exploits reward model weaknesses")
    print("  • Distribution shift: Training prompts ≠ deployment prompts")
    print("  • Value alignment: Hard to specify what we want")

visualize_rlhf_pipeline()

## 5. Reward Model

**Reward Model** predicts human preferences

**Training Data**: Comparisons
- Prompt: "Explain quantum computing"
- Response A: "Quantum computers use qubits..."
- Response B: "I don't know"
- Preference: A > B

**Bradley-Terry Model**: Model probability of preferring A over B

$$P(y_A \succ y_B) = \frac{\exp(r(x, y_A))}{\exp(r(x, y_A)) + \exp(r(x, y_B))} = \sigma(r(x, y_A) - r(x, y_B))$$

**Loss**:

$$L = -\log \sigma(r(x, y_w) - r(x, y_l))$$

Where $y_w$ is preferred (winner), $y_l$ is rejected (loser)

In [ ]:
class RewardModel(nn.Module):
    """
    Reward model for RLHF.
    
    Predicts scalar reward for (prompt, response) pair.
    """
    
    def __init__(self, hidden_dim: int = 256):
        super().__init__()
        
        # In practice, this would be based on pretrained LLM
        # Here we use simple encoder for demonstration
        self.encoder = nn.Sequential(
            nn.Linear(512, hidden_dim),  # [CLS] embedding from LLM
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        
        # Reward head
        self.reward_head = nn.Linear(hidden_dim, 1)
    
    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (batch, 512) embeddings from LLM
            
        Returns:
            rewards: (batch,) scalar rewards
        """
        features = self.encoder(embeddings)
        rewards = self.reward_head(features).squeeze(-1)
        return rewards

def compute_reward_loss(reward_model: RewardModel,
                        chosen_embeddings: torch.Tensor,
                        rejected_embeddings: torch.Tensor) -> torch.Tensor:
    """
    Compute reward model loss using Bradley-Terry model.
    
    Args:
        reward_model: Reward model
        chosen_embeddings: Embeddings for preferred responses
        rejected_embeddings: Embeddings for rejected responses
        
    Returns:
        loss: Bradley-Terry loss
    """
    # Compute rewards
    r_chosen = reward_model(chosen_embeddings)
    r_rejected = reward_model(rejected_embeddings)
    
    # Bradley-Terry loss
    # Loss = -log(sigmoid(r_chosen - r_rejected))
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    
    # Accuracy (for monitoring)
    with torch.no_grad():
        accuracy = ((r_chosen > r_rejected).float().mean())
    
    return loss, accuracy

# Example reward model
print("\nReward Model:")
print("="*70)

reward_model = RewardModel(hidden_dim=256).to(device)
print(reward_model)

# Simulate preference data
batch_size = 32
chosen_emb = torch.randn(batch_size, 512).to(device)
rejected_emb = torch.randn(batch_size, 512).to(device)

loss, accuracy = compute_reward_loss(reward_model, chosen_emb, rejected_emb)

print(f"\nSimulated Training Step:")
print(f"  Batch size: {batch_size}")
print(f"  Loss: {loss.item():.4f}")
print(f"  Accuracy: {accuracy.item():.2%}")

print("\nReward Model Training:")
print("  • Data: Thousands of human preference comparisons")
print("  • Base: Usually initialize from SFT model")
print("  • Training: Fine-tune on preference data")
print("  • Evaluation: Accuracy on held-out comparisons")
print("  • Challenge: Reward model quality critical for RLHF")

## 6. DPO - Direct Preference Optimization

**DPO** is a simpler alternative to RLHF (no reward model, no RL!)

**Key Insight**: Optimal RL policy has closed form:

$$\pi^*(y|x) = \frac{1}{Z(x)}\pi_{\text{ref}}(y|x)\exp\left(\frac{r(x,y)}{\beta}\right)$$

**DPO reparameterizes** reward in terms of policy:

$$r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta\log Z(x)$$

**DPO Loss**: Directly optimize policy on preferences

$$L_{\text{DPO}} = -\mathbb{E}_{(x,y_w,y_l)}\left[\log\sigma\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

**Benefits**:
- Simpler: No reward model, no PPO
- Stable: Direct supervised learning
- Effective: Competitive with RLHF

In [ ]:
def compute_dpo_loss(policy_logprobs_chosen: torch.Tensor,
                     policy_logprobs_rejected: torch.Tensor,
                     reference_logprobs_chosen: torch.Tensor,
                     reference_logprobs_rejected: torch.Tensor,
                     beta: float = 0.1) -> Tuple[torch.Tensor, Dict]:
    """
    Compute DPO loss.
    
    Args:
        policy_logprobs_chosen: Log probs from policy for chosen responses
        policy_logprobs_rejected: Log probs from policy for rejected responses
        reference_logprobs_chosen: Log probs from reference for chosen
        reference_logprobs_rejected: Log probs from reference for rejected
        beta: Temperature parameter
        
    Returns:
        loss: DPO loss
        metrics: Dict of metrics
    """
    # Log ratios
    policy_logratios = policy_logprobs_chosen - policy_logprobs_rejected
    reference_logratios = reference_logprobs_chosen - reference_logprobs_rejected
    
    # DPO loss
    logits = beta * (policy_logratios - reference_logratios)
    loss = -F.logsigmoid(logits).mean()
    
    # Metrics
    with torch.no_grad():
        # Accuracy
        accuracy = (logits > 0).float().mean()
        
        # Implicit rewards
        chosen_rewards = beta * (policy_logprobs_chosen - reference_logprobs_chosen)
        rejected_rewards = beta * (policy_logprobs_rejected - reference_logprobs_rejected)
        reward_margin = (chosen_rewards - rejected_rewards).mean()
    
    metrics = {
        'accuracy': accuracy.item(),
        'reward_margin': reward_margin.item(),
        'chosen_reward_mean': chosen_rewards.mean().item(),
        'rejected_reward_mean': rejected_rewards.mean().item()
    }
    
    return loss, metrics

# Visualize DPO vs RLHF
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RLHF pipeline
ax1 = axes[0]
ax1.set_title('RLHF Pipeline', fontsize=14, weight='bold')
ax1.axis('off')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

rlhf_steps = [
    (0.5, 0.85, 'SFT Model', 'lightblue'),
    (0.5, 0.65, '1. Train Reward\nModel', 'lightgreen'),
    (0.5, 0.45, '2. PPO Training', 'lightyellow'),
    (0.5, 0.25, 'Aligned Model', 'lightcoral')
]

for x, y, text, color in rlhf_steps:
    rect = plt.Rectangle((x-0.15, y-0.06), 0.3, 0.12,
                         facecolor=color, edgecolor='black', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(x, y, text, ha='center', va='center',
            fontsize=11, weight='bold')

# Arrows
for i in range(len(rlhf_steps)-1):
    _, y1, _, _ = rlhf_steps[i]
    _, y2, _, _ = rlhf_steps[i+1]
    ax1.annotate('', xy=(0.5, y2+0.06), xytext=(0.5, y1-0.06),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax1.text(0.5, 0.08, 'Complex: 2 stages, RL training', ha='center',
        fontsize=10, style='italic', color='red')

# DPO pipeline
ax2 = axes[1]
ax2.set_title('DPO Pipeline', fontsize=14, weight='bold')
ax2.axis('off')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

dpo_steps = [
    (0.5, 0.75, 'SFT Model', 'lightblue'),
    (0.5, 0.45, 'DPO Training\n(Direct on\nPreferences)', 'lightgreen'),
    (0.5, 0.25, 'Aligned Model', 'lightcoral')
]

for x, y, text, color in dpo_steps:
    rect = plt.Rectangle((x-0.15, y-0.08), 0.3, 0.16,
                         facecolor=color, edgecolor='black', linewidth=2)
    ax2.add_patch(rect)
    ax2.text(x, y, text, ha='center', va='center',
            fontsize=11, weight='bold')

# Arrows
for i in range(len(dpo_steps)-1):
    _, y1, _, _ = dpo_steps[i]
    _, y2, _, _ = dpo_steps[i+1]
    ax2.annotate('', xy=(0.5, y2+0.08), xytext=(0.5, y1-0.08),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax2.text(0.5, 0.08, 'Simple: 1 stage, supervised learning', ha='center',
        fontsize=10, style='italic', color='green')

plt.tight_layout()
plt.show()

print("\nDPO vs RLHF:")
print("="*70)

print("\nRLHF:")
print("  ✓ Well-established (ChatGPT, Claude)")
print("  ✓ Flexible reward modeling")
print("  ✗ Complex: Reward model + PPO training")
print("  ✗ Unstable: RL can be tricky")
print("  ✗ Expensive: Multiple models and training stages")

print("\nDPO:")
print("  ✓ Simple: Direct supervised learning")
print("  ✓ Stable: No RL")
print("  ✓ Efficient: Single model, one training stage")
print("  ✓ Competitive performance with RLHF")
print("  ✗ Newer: Less battle-tested")

print("\nWhen to use DPO:")
print("  • You have preference data")
print("  • You want simpler training")
print("  • You need more stable training")
print("  • You have limited compute")

# Example DPO computation
print("\nExample DPO computation:")
batch_size = 8
policy_chosen = torch.randn(batch_size).to(device)
policy_rejected = torch.randn(batch_size).to(device)
ref_chosen = torch.randn(batch_size).to(device)
ref_rejected = torch.randn(batch_size).to(device)

loss, metrics = compute_dpo_loss(
    policy_chosen, policy_rejected,
    ref_chosen, ref_rejected,
    beta=0.1
)

print(f"  Loss: {loss.item():.4f}")
print(f"  Accuracy: {metrics['accuracy']:.2%}")
print(f"  Reward margin: {metrics['reward_margin']:.4f}")

## Summary

You've mastered advanced RL and LLM alignment:
- ✅ PPO with clipped objective
- ✅ GAE for advantage estimation
- ✅ RLHF pipeline (SFT → Reward Model → PPO)
- ✅ Reward modeling from human preferences
- ✅ DPO as simpler RLHF alternative

**Key Insights**:
1. **PPO** is the workhorse of modern RL (games, robotics, LLMs)
2. **Clipping** prevents destructive policy updates
3. **GAE** balances bias-variance in advantage estimation
4. **RLHF** aligns LLMs with human preferences (ChatGPT, Claude)
5. **DPO** simplifies alignment (no RM, no RL)

**Algorithm Comparison**:

| Method | Complexity | Stability | Data Efficiency | Performance |
|--------|-----------|-----------|-----------------|-------------|
| REINFORCE | Low | Low | Low | Medium |
| Actor-Critic | Medium | Medium | Medium | Good |
| PPO | Medium | High | High | Excellent |
| RLHF (PPO) | High | Medium | Medium | Excellent |
| DPO | Low | High | High | Excellent |

**Real-World Applications**:
- **ChatGPT**: RLHF with PPO for instruction following
- **Claude**: Constitutional AI + RLHF for safety
- **GitHub Copilot**: Fine-tuned on code with RLHF
- **Robotics**: PPO for continuous control
- **Game AI**: PPO for Dota 2, StarCraft II

**Future Directions**:
- Multi-objective RLHF (helpfulness + safety + ...) 
- Scalable oversight (AI-assisted evaluation)
- Offline RL for safety
- Constitutional AI (self-critique and revision)

## Exercises

1. **Implement full PPO**: Train PPO agent on CartPole-v1
2. **Compare GAE lambdas**: Train with λ = [0.0, 0.5, 0.95, 1.0], compare performance
3. **Build reward model**: Create dataset of preferences, train reward model
4. **Simulate RLHF**: Implement simplified RLHF pipeline on text generation
5. **DPO implementation**: Implement full DPO training on preference dataset
6. **Analyze KL penalty**: Study effect of KL coefficient β on policy updates